In [1]:
from src.data import DataLoader, DataPreprocessor
from src.features import ProductFeatureExtractor, CustomerFeatureExtractor, TrainingDataBuilder
from src.training import TemporalDataSplitter
from src.models import (
    PopularityRecommender, 
    PersonalFrequencyRecommender, 
    LightGBMRanker,
    XGBoostRanker  # if installed
)
from src.evaluation import RankingMetrics
from src.inference import RecommenderPredictor


CatBoost not installed - pip install catboost


In [ ]:
# 1. Load & preprocess
raw = DataLoader().load_csv("src\data\data_raw.csv")
prepared = DataPreprocessor(min_orders=2).transform(raw.transactions)

In [ ]:
# 2. Features
product_features = ProductFeatureExtractor().extract(prepared)
customer_profiles = CustomerFeatureExtractor().extract(prepared)
training_data = TrainingDataBuilder().build(prepared, product_features, customer_profiles)


In [ ]:
# 3. Split
split = TemporalDataSplitter(test_ratio=0.2).split(training_data)

In [ ]:
# 4. Model Selection
evaluator = RankingMetrics(k_values=[1, 3, 5, 10])

# Define candidate models
candidate_models = [
    ("Popularity", PopularityRecommender()),
    ("PersonalFreq_s0.2", PersonalFrequencyRecommender(smoothing=0.2)),
    ("PersonalFreq_s0.3", PersonalFrequencyRecommender(smoothing=0.3)),
    ("PersonalFreq_s0.5", PersonalFrequencyRecommender(smoothing=0.5)),
    ("LightGBM", LightGBMRanker(num_leaves=31, learning_rate=0.05)),
    ("XGBoost", XGBoostRanker(max_depth=6, learning_rate=0.1))
]

# Train and evaluate all models
results = []
for name, model in candidate_models:
    print(f"Training {name}...")
    
    # Train
    if hasattr(model, 'name') and 'LightGBM' in model.name:
        model.fit(split.train_df, split.feature_names, split.test_df)  # with validation
    else:
        model.fit(split.train_df, split.feature_names)
    
    # Evaluate
    metrics = evaluator.evaluate(model, split.test_df, split.feature_names)
    
    results.append({
        'name': name,
        'model': model,
        'ndcg@3': metrics['ndcg@3'],
        'hit_rate@3': metrics['hit_rate@3'],
        'metrics': metrics
    })
    
    print(f"  NDCG@3: {metrics['ndcg@3']:.4f}, Hit@3: {metrics['hit_rate@3']:.4f}")


In [ ]:
# 5. Evaluate
results.sort(key=lambda x: x['ndcg@3'], reverse=True)

print("\n" + "=" * 60)
print("MODEL COMPARISON (sorted by NDCG@3)")
print("=" * 60)
print(f"{'Model':<25} {'NDCG@3':<10} {'Hit@3':<10}")
print("-" * 60)
for r in results:
    print(f"{r['name']:<25} {r['ndcg@3']:<10.4f} {r['hit_rate@3']:<10.4f}")
print("=" * 60)

# Best model
best = results[0]
best_model = best['model']
print(f"\n🏆 Best Model: {best['name']} (NDCG@3 = {best['ndcg@3']:.4f})")

# Also keep best baseline for cold-start fallback
baseline = next(r['model'] for r in results if 'PersonalFreq' in r['name'])

In [ ]:
# 6. Recommend
predictor = RecommenderPredictor(best_model, baseline, product_features, prepared)
result = predictor.recommend(customer_id=5, top_k=5)
print(predictor.format_prediction(result))

In [2]:
loader = DataLoader()

raw_data = loader.load_from_bigquery(
    project_id="jr-data-training",
    query="""
        SELECT * FROM `jr-data-training.cafe.cafe-sales`
    """
)


c:\Users\laaro\Downloads\jr\data-training\.venv\Lib\site-packages\google\auth\_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
c:\Users\laaro\Downloads\jr\data-training\.venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [4]:
raw_data

RawData(transactions=       order_id  customer_id                                   ip_addr  \
0         26363          256  acebe24917d7531eee22c76aad6607c4eef06d90   
1         23636          256  020e00e2fe8505632ee3e3ad08084c5366980e2a   
2         19942          256  fd024808d9a1f3f5905a70afd1d8d6251245e496   
3         20878          256  ed55e18c0b383738460a00c184e16e68df7b8018   
4         20623          256  ed55e18c0b383738460a00c184e16e68df7b8018   
...         ...          ...                                       ...   
37367     31184         1279  846353460de3c14fe43b4e59c153a3c70663b2d8   
37368     31036         1279  69ee19df9d9bd5b47f736886c2d3ed1858c1c9cc   
37369     30976         1279  69ee19df9d9bd5b47f736886c2d3ed1858c1c9cc   
37370     29921         1279  69ee19df9d9bd5b47f736886c2d3ed1858c1c9cc   
37371     33457         1535  5364efbd8ec01c308ca49d14c69648161483de0e   

             date_created           date_paid  total  status  \
0     2022-10-30 10:27:12 